[Back to Computer Networks guideline](Computer-Networks.html)


## **Internet Architecture and the Life of a Packet**

A browser makes networking look like a single action: enter a URL and receive a page. Underneath that action, many independently designed systems cooperate. The computer first needs local connectivity, a name must become one or more destination addresses, packets must cross networks owned by different organizations, and the remote endpoint must prove its identity before application data can be trusted.

This chapter builds a mental model for that entire path. The running example is an HTTPS request, but the purpose is broader: to learn **where each networking responsibility lives**, **what state each component keeps**, and **which quantity explains a performance problem**. Those distinctions are more useful than memorizing a list of protocol names.

::: {.callout-note}
The Internet and the Web are not synonyms. The **Internet** is the packet-delivery infrastructure and protocol ecosystem. The **Web** is one application family built on top of it, mainly using URLs, HTTP, HTML, and browser technologies.
:::

By the end of the chapter, a statement such as "the network is slow" should feel incomplete. A slow interaction may come from DNS resolution, connection setup, propagation distance, a congested queue, a low-rate access link, server computation, or repeated application requests. Each cause needs different evidence and a different remedy.


### **What Is a Computer Network?**

#### **Communication, Resource Sharing, and Scale**

A **computer network** is a set of autonomous computing devices that exchange data through communication links according to shared protocols. "Autonomous" matters: unlike cores inside one processor, networked hosts have separate clocks, memories, failure modes, administrators, and security boundaries. No host can directly inspect the complete state of every other host or link.

A useful analogy is a logistics system. Applications create messages as customers create shipments. Protocol headers act like labels that identify how a shipment should be handled. Links are transport segments, switches and routers are transfer facilities, and queues appear when arrivals temporarily exceed available service capacity. The analogy is imperfect, but it captures an important property: the network moves **bounded units** through multiple independently operated stages rather than creating one permanent wire between every pair of applications.

Networks solve three recurring problems:

| Need | What networking provides | Example |
|---|---|---|
| Communication | A way for separate processes to exchange bytes or messages | A browser requests an object from a web server |
| Resource sharing | Many users can reach shared compute, storage, printers, or services | Thousands of clients use one cloud API |
| Scale and composition | Small networks can interconnect without one global owner | A home Wi-Fi network reaches a university service through several ISPs |

Why not connect every pair of hosts with a dedicated link? With $n$ hosts, a full mesh requires $n(n-1)/2$ links. That grows quadratically, wastes capacity when pairs are idle, and becomes impossible to operate across organizations. Shared links, switching, hierarchical addressing, and routing let the system scale while allowing paths to be selected dynamically.

#### **Hosts, Links, Switches, and Routers**

The main physical and logical roles are:

| Component | Primary responsibility | Usually examines |
|---|---|---|
| **Host / end system** | Runs applications and originates or consumes end-to-end traffic | Application, transport, IP, and link information |
| **Link** | Carries a signal between adjacent interfaces | Bits or symbols on copper, fiber, or radio |
| **Switch** | Forwards frames within a link-layer domain such as an Ethernet LAN | Link-layer addresses and VLAN information |
| **Router** | Forwards IP datagrams between networks | Destination IP prefix and forwarding state |
| **Middlebox** | Performs an additional function such as NAT, filtering, proxying, or load balancing | Depends on the function; often several layers |

The boundaries are functional rather than purely physical. A home "router" often contains a Wi-Fi access point, Ethernet switch, IP router, DHCP server, firewall, and NAT in one device. Separating those roles mentally makes its behavior much easier to diagnose.

#### **Network Edge, Access Network, and Core**

The **network edge** contains hosts and the applications that create value for users. The **access network** connects an edge device to its first router through technologies such as Ethernet, Wi-Fi, passive optical networking, cable, or cellular radio. The **network core** is the mesh of high-capacity routers and links that forwards traffic between access networks.

This edge/access/core division explains why two users of the same application can see very different performance. Their destination may be identical, while their radio conditions, home queue, access rate, ISP path, or geographic distance differ. It also foreshadows the end-to-end principle: many correctness guarantees need knowledge available only at the endpoints, while the core is optimized for forwarding packets at scale.


### **The Structure of the Internet**

#### **ISPs, Autonomous Systems, and Administrative Boundaries**

The Internet is not one network operated by one company. It is an interconnection of networks that agree to carry IP traffic. An **Internet Service Provider (ISP)** sells or supplies connectivity, but many non-ISP organizations also operate substantial networks: universities, enterprises, cloud providers, content delivery networks, and governments.

For inter-domain routing, a network is represented as one or more **Autonomous Systems (ASes)**. An AS is an administrative and routing-policy boundary identified by an Autonomous System Number (ASN). It is not simply "a large router" or "one physical network." The operator decides which routes to advertise, which neighbors to use, and which traffic relationships are economically acceptable.

Inside an AS, the operator can optimize paths according to its own engineering goals. Between ASes, reachability is exchanged with BGP and is strongly influenced by policy and business relationships. Consequently, an Internet path is not guaranteed to be geographically shortest, symmetric, or permanently stable.

#### **Internet Exchange Points and Peering**

Networks commonly interconnect through two economic relationships:

| Relationship | Meaning | Typical consequence |
|---|---|---|
| **Transit** | One network pays another to provide reachability to the wider Internet | The provider advertises many external destinations |
| **Peering** | Two networks exchange traffic for their respective customers, often without per-byte payment | Traffic can avoid an upstream transit path |

An **Internet Exchange Point (IXP)** provides shared physical infrastructure where many networks can establish peering sessions. An IXP does not automatically make every participant a peer of every other participant; the routing relationships still depend on policy and agreement. Its value is that networks can interconnect locally, reduce transit cost, and often reduce latency.

#### **The Internet as a Network of Networks**

The following open-license diagram captures the key structure: end users attach through access networks; ISPs and autonomous systems exchange traffic through private connections or IXPs; no single box is "the Internet."

![Structure of the Internet, including access networks, ISPs, autonomous systems, peer connections, and an Internet exchange.](assets/structure-of-the-internet.svg){fig-alt="Structure of the Internet with access networks, ISPs, autonomous systems, peer connections, and an Internet exchange" width="92%"}

*Figure source: [BWenk, Structure of the Internet, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Structure_of_the_Internet.svg), licensed under CC BY-SA 4.0.*

A packet therefore crosses two kinds of boundaries at once. At each **hop**, a router makes a local forwarding decision. At each **administrative boundary**, independent organizations apply routing, security, and traffic-engineering policy. The resulting path is an emergent outcome of many local decisions rather than a route computed by a global Internet controller.


### **Protocols, Services, and Interfaces**

#### **Protocol Syntax, Semantics, and Timing**

A **protocol** defines the rules followed by communicating entities. A complete protocol description needs more than a packet layout:

- **Syntax** specifies representation: field order, field size, encoding, and message format.
- **Semantics** specifies meaning: what a field requests, reports, acknowledges, or rejects.
- **Timing** specifies behavior over time: who sends first, what event triggers a reply, and what happens after a timeout.

For example, a TCP segment has a defined binary syntax. A SYN flag has connection-establishment semantics. Retransmission timers and acknowledgment behavior define part of TCP's timing. Two implementations that agree only on field positions but disagree on state transitions still cannot interoperate.

#### **Service Models and Programming Interfaces**

A **service** describes what one layer offers to the layer above; a **protocol** describes how peer entities cooperate to provide that service; an **interface** describes how a local program accesses it. These ideas are related but not interchangeable.

Consider a browser using TCP. The service is an ordered, reliable byte stream between endpoints. TCP is the peer protocol that uses sequence numbers, acknowledgments, timers, and flow control to provide that abstraction. The socket API is the local programming interface through which the browser requests the service. A service answers "what can I rely on?"; a protocol answers "how do peers coordinate?"; an API answers "how does software invoke it?"

#### **Layering, Encapsulation, and Multiplexing**

Layering controls complexity by assigning each layer a narrower contract. When an application sends data, each lower layer adds control information in a process called **encapsulation**:

1. The application creates a message, such as an HTTP request.
2. The transport layer adds endpoint and transport state, producing a TCP segment or UDP datagram.
3. The Internet layer adds source and destination IP information, producing an IP datagram.
4. The link layer places that datagram in a frame for one local hop.
5. The physical layer encodes the frame as signals.

At the receiver, headers are interpreted and removed in reverse order. Encapsulation is not merely vocabulary: it explains why a router can forward an encrypted HTTPS exchange without understanding the web page, and why an Ethernet address usually changes at every routed hop while the destination IP address normally remains end-to-end.

![Application data encapsulated inside a UDP datagram, an IP datagram, and a link-layer frame.](assets/udp-encapsulation.svg){fig-alt="Application data is wrapped by UDP, IP, and link-layer headers" width="82%"}

*Figure source: [Cburnett and Kbrose, UDP encapsulation, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:UDP_encapsulation.svg), licensed under CC BY-SA 3.0/GFDL.*

The reverse operation is **demultiplexing**. A link-layer type identifies IPv4 or IPv6; an IP next-header or protocol field identifies TCP, UDP, or another transport; a transport port helps deliver data to the appropriate socket; application framing identifies messages within the byte stream. Each layer needs enough metadata to choose the next consumer.

#### **TCP/IP and OSI Models**

The OSI model is a seven-layer reference vocabulary. The deployed Internet protocol suite is more naturally described with four or five layers. The mapping is approximate because real systems are not obliged to fit a pedagogical diagram perfectly.

| Internet layer | Approximate OSI layers | Representative protocols or technologies | Data unit |
|---|---|---|---|
| Application | Application, presentation, session | HTTP, DNS, TLS, SSH | Message / application data |
| Transport | Transport | TCP, UDP, QUIC's transport functions | Segment or datagram |
| Internet | Network | IPv4, IPv6, ICMP | IP datagram / packet |
| Link | Data link | Ethernet, Wi-Fi, PPP | Frame |
| Physical | Physical | Copper, fiber, radio | Bits / symbols |

The figure below makes the scope distinction explicit. Applications and transport protocols operate end to end. IP is processed by hosts and routers. A link protocol operates only between adjacent interfaces and can change at every hop.

![Two hosts communicate through two routers, showing which Internet protocol layers operate at each device.](assets/ip-stack-connections.svg){fig-alt="Two hosts and two routers with the Internet protocol layers used at each hop" width="68%"}

*Figure source: [Kbrose, IP stack connections, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:IP_stack_connections.svg), licensed under CC BY-SA 3.0/GFDL.*

Layering is a tool, not a law. Cross-layer information can improve performance, and technologies such as QUIC combine transport, security, and application-facing functions. The design question is whether a dependency creates enough benefit to justify tighter coupling and more difficult evolution.


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class LayerOverhead:
    """Header/trailer bytes added by one simplified protocol layer."""

    name: str
    bytes_added: int


def encapsulation_report(payload: bytes) -> list[dict[str, int | str]]:
    """Show how a payload grows while moving down a typical UDP/IPv4 stack.

    The sizes use minimum UDP and IPv4 headers and a basic Ethernet II
    header plus frame check sequence. Preamble and inter-frame gap are not
    included because they are physical-link overhead rather than frame bytes.
    """

    layers = [
        LayerOverhead("Application payload", 0),
        LayerOverhead("UDP header", 8),
        LayerOverhead("IPv4 header", 20),
        LayerOverhead("Ethernet header + FCS", 18),
    ]

    cumulative = len(payload)
    report = []
    for layer in layers:
        cumulative += layer.bytes_added
        report.append(
            {
                "stage": layer.name,
                "added_bytes": layer.bytes_added,
                "cumulative_bytes": cumulative,
            }
        )
    return report


message = b"temperature=21.7C"
for row in encapsulation_report(message):
    print(
        f"{row['stage']:<24} +{row['added_bytes']:>2} bytes"
        f" -> {row['cumulative_bytes']:>3} bytes"
    )


### **Packet Switching**

#### **Packets, Datagrams, and Store-and-Forward**

A long application message is divided into bounded units so that links and buffers can be shared. **Packet** is the general term. An **IP datagram** is an Internet-layer packet designed to be forwarded independently. A **frame** is the link-layer unit used on one hop. The same bytes may therefore be described differently depending on the layer being discussed.

Most Internet routers use **store-and-forward** behavior: the router receives enough of a packet to validate and classify it, stores it in memory, chooses an output, and transmits it on the next link. For a packet of $L$ bits on a link with rate $R$ bits/s, serialization takes

$$
d_{trans} = \frac{L}{R}.
$$

This is **transmission delay**, the time required to push all packet bits onto a link. It does not depend on the physical length of the link. A 12,000-bit packet takes $120\ \mu s$ to serialize at 100 Mb/s whether the cable is one metre or one hundred kilometres long.

![Animation of packets taking different paths through a packet-switched network.](assets/packet-switching.gif){fig-alt="Animated packet switching example with packets forwarded through intermediate nodes" width="86%"}

*Figure source: [Oddbodz, Packet Switching, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Packet_Switching.gif), licensed under CC BY-SA 3.0.*

If $N$ equal packets cross $H$ equal-rate links, with no propagation, processing, or queueing delay, the first packet needs $HL/R$ to reach the destination. Once the pipeline is full, later packets can follow one serialization interval apart:

$$
T_{N,H} = (H + N - 1)\frac{L}{R}.
$$

The formula exposes why packetization helps: different links can work on different packets simultaneously. It also exposes a cost: every packet needs headers and processing, so extremely small packets waste capacity.

#### **Statistical Multiplexing**

Internet traffic is bursty. A user may download rapidly for a short period and then remain silent while reading. **Statistical multiplexing** allocates link capacity to packets that are actually present rather than reserving a fixed slice for each potential sender. When only one flow is active, it may use nearly the whole link; when many are active, their packets are interleaved.

This improves utilization but removes deterministic guarantees. Several bursts can arrive together, creating a queue. The design trades fixed reservations for efficient sharing and uses buffering, congestion control, scheduling, and application adaptation to manage contention.

#### **Packet Switching vs Circuit Switching**

| Property | Packet switching | Circuit switching |
|---|---|---|
| Resource allocation | Shared dynamically among active packets | Reserved for a session or circuit |
| Bursty workloads | Usually efficient | Reserved capacity may sit idle |
| Delay | Variable because queues can form | More predictable after setup |
| Admission | A packet can usually be offered immediately | A circuit may be rejected if capacity is unavailable |
| Failure response | Routing can move later packets to another path | Circuit must be repaired or re-established |
| Typical fit | General-purpose data networks | Workloads needing strict reserved service |

Neither design is universally better. Packet switching matches diverse, bursty Internet applications. Reservation remains useful when a system must provide a hard service guarantee and has admission control to protect that guarantee.

#### **Queues, Buffers, and Packet Loss**

An output queue grows when packets arrive faster than the outgoing link can serialize them. Let $\lambda$ be the average packet arrival rate, $L$ the average packet size in bits, and $R$ the output rate. The offered load is

$$
\rho = \frac{\lambda L}{R}.
$$

When $\rho$ approaches one, small bursts can create large queueing delay. If long-term offered load exceeds capacity, $\rho > 1$, no finite buffer can prevent persistent growth. Once the buffer is full, new or selected queued packets are dropped. More buffering can reduce short-term loss, but an oversized unmanaged queue can create **bufferbloat**: packets survive while interactive latency becomes very large.

Queueing is therefore not an accidental implementation detail. It is where competing traffic meets a finite resource, and its behavior influences latency, fairness, loss, and congestion signals.


In [ ]:
def packet_pipeline_time(
    packet_bytes: int,
    link_rates_mbps: list[float],
    packet_count: int,
) -> dict[str, float]:
    """Calculate an ideal store-and-forward pipeline completion time.

    Assumptions:
    1. Every packet has the same size.
    2. Links are initially idle and have infinite buffers.
    3. Processing and propagation delays are ignored.
    4. A router forwards a packet only after receiving all of it.
    """

    packet_bits = packet_bytes * 8
    serialization = [
        packet_bits / (rate_mbps * 1_000_000)
        for rate_mbps in link_rates_mbps
    ]

    # The first packet must be serialized once on every link.
    first_packet_seconds = sum(serialization)

    # Later packets leave at the pace of the slowest (bottleneck) link.
    pipeline_spacing_seconds = max(serialization)
    total_seconds = first_packet_seconds + (packet_count - 1) * pipeline_spacing_seconds

    return {
        "first_packet_ms": first_packet_seconds * 1_000,
        "bottleneck_spacing_ms": pipeline_spacing_seconds * 1_000,
        "all_packets_ms": total_seconds * 1_000,
    }


result = packet_pipeline_time(
    packet_bytes=1_500,
    link_rates_mbps=[10, 100, 20],
    packet_count=5,
)

for metric, value in result.items():
    print(f"{metric:<25} {value:8.3f}")


### **Network Performance**

#### **Processing, Queueing, Transmission, and Propagation Delay**

For one packet at one router, **nodal delay** is commonly decomposed as

$$
d_{nodal} = d_{proc} + d_{queue} + d_{trans} + d_{prop}.
$$

- $d_{proc}$ is processing time: validate fields, classify the packet, and choose an output.
- $d_{queue}$ is time waiting for earlier packets. It changes with traffic and scheduling.
- $d_{trans}=L/R$ is serialization time for $L$ bits on a rate-$R$ link.
- $d_{prop}=D/s$ is propagation time across distance $D$ at signal speed $s$ in the medium.

![Processing, queueing, transmission, and propagation as consecutive components of one packet's nodal delay.](assets/node-delay-components.svg){fig-alt="A packet is processed, waits in a queue, is serialized, and propagates to the next router" width="94%"}

The most common confusion is between transmission and propagation. Increasing link rate reduces $L/R$ but cannot make a signal exceed the propagation speed of the medium. Moving a service geographically closer reduces $D/s$ but does not fix a low-rate access link. A performance diagnosis must identify the relevant term before proposing a remedy.

For a path with several links, a simplified one-way delay is the sum of the per-hop terms:

$$
d_{path} \approx \sum_{i=1}^{H}
\left(d_{proc,i}+d_{queue,i}+\frac{L}{R_i}+\frac{D_i}{s_i}\right).
$$

Real traffic complicates this expression through packet-size variation, operating-system scheduling, wireless retransmissions, route changes, transport recovery, and application dependencies. The decomposition remains valuable because it tells us what to measure.

#### **Round-Trip Time and Jitter**

**Round-trip time (RTT)** measures elapsed time from sending a request or probe until the corresponding response or acknowledgment returns. RTT is not necessarily twice one-way delay: forward and reverse routes can differ, and endpoint processing can be asymmetric.

**Jitter** describes variation in packet delay. Two paths can have the same average latency but very different user experience. Interactive audio and gaming are sensitive to jitter because late data may be useless even if it eventually arrives. Receivers can hide some variation with a playout buffer, trading additional fixed latency for smoother delivery.

#### **Throughput, Bottlenecks, and Goodput**

**Bandwidth** or link capacity is a configured or physical rate. **Throughput** is the rate actually delivered over an interval. On a serial path, sustained throughput cannot exceed the slowest effective resource:

$$
Throughput \leq \min(R_1,R_2,\ldots,R_H).
$$

The minimum-rate link is a **bottleneck**, but observed throughput may be lower because of competing flows, protocol overhead, receiver limits, congestion control, loss recovery, or application behavior.

**Goodput** counts useful application payload delivered per unit time. It excludes protocol headers, retransmitted bytes, and sometimes duplicated or unusable data. A link can be busy at high throughput while application goodput is poor; repeated retransmissions are the classic example.

#### **Bandwidth-Delay Product and Little's Law**

The **bandwidth-delay product (BDP)** estimates how much data can be in flight on a path:

$$
BDP = R \times RTT.
$$

At 100 Mb/s with a 40 ms RTT, the BDP is $4,000,000$ bits, or about 500 kB. A reliable transport needs enough permitted in-flight data to fill that path. If its sending window is only 64 kB, it cannot sustain 100 Mb/s regardless of the raw link rate.

**Little's Law** relates average occupancy $N$, average arrival rate $\lambda$, and average time in the system $W$:

$$
N = \lambda W.
$$

For a stable queue, if 2,000 packets/s pass through and a packet spends an average of 5 ms in that queue, the average occupancy is $2{,}000 \times 0.005 = 10$ packets. The law does not explain why the delay exists, but it gives a powerful consistency check between rate, residence time, and amount of work in flight.


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Link:
    rate_mbps: float
    distance_km: float
    propagation_speed_m_s: float = 2.0e8  # Approximate speed in fiber/copper.


def estimate_path(
    packet_bytes: int,
    links: list[Link],
    processing_ms_per_hop: float = 0.05,
    queueing_ms_per_hop: float = 0.0,
) -> dict[str, float]:
    """Estimate one-way packet delay and related path quantities."""

    packet_bits = packet_bytes * 8
    transmission_s = sum(
        packet_bits / (link.rate_mbps * 1_000_000)
        for link in links
    )
    propagation_s = sum(
        (link.distance_km * 1_000) / link.propagation_speed_m_s
        for link in links
    )
    processing_s = len(links) * processing_ms_per_hop / 1_000
    queueing_s = len(links) * queueing_ms_per_hop / 1_000

    one_way_s = transmission_s + propagation_s + processing_s + queueing_s
    estimated_rtt_s = 2 * one_way_s
    bottleneck_bps = min(link.rate_mbps for link in links) * 1_000_000
    bdp_bytes = bottleneck_bps * estimated_rtt_s / 8

    return {
        "transmission_ms": transmission_s * 1_000,
        "propagation_ms": propagation_s * 1_000,
        "processing_ms": processing_s * 1_000,
        "queueing_ms": queueing_s * 1_000,
        "one_way_ms": one_way_s * 1_000,
        "estimated_rtt_ms": estimated_rtt_s * 1_000,
        "bottleneck_mbps": bottleneck_bps / 1_000_000,
        "bdp_kib": bdp_bytes / 1024,
    }


path = [
    Link(rate_mbps=100, distance_km=2),
    Link(rate_mbps=1_000, distance_km=1_200),
    Link(rate_mbps=200, distance_km=20),
]

metrics = estimate_path(
    packet_bytes=1_500,
    links=path,
    processing_ms_per_hop=0.08,
    queueing_ms_per_hop=1.2,
)

for name, value in metrics.items():
    print(f"{name:<22} {value:9.3f}")


### **Internet Design Principles**

#### **Best-Effort Delivery and the Narrow Waist**

IP offers a deliberately limited **best-effort datagram service**. It attempts delivery but does not promise that a packet will arrive, arrive once, arrive in order, or arrive within a deadline. Those missing guarantees are not evidence that IP is unfinished. A small common service can run over many link technologies and support many transports and applications.

This is the Internet's **narrow waist** or hourglass architecture. Above IP are many applications and transport choices. Below IP are Ethernet, Wi-Fi, fiber, cellular, satellite, and future link technologies. The common IP layer lets innovation happen on either side without requiring every application to understand every physical network.

The benefit is interoperability and evolvability. The cost is that applications cannot infer strong performance or reliability guarantees from IP alone. They must choose an appropriate transport, cope with failure, and measure what matters to their users.

#### **The End-to-End Principle**

The end-to-end argument asks where a function can be implemented **completely and correctly**. If correctness requires application knowledge available only at the endpoints, implementing the function solely inside the network cannot remove the endpoint's responsibility.

Reliable file transfer is a classic example. Link checksums and local retries reduce corruption and improve performance, but only the sender and receiver can verify that the intended complete file was stored correctly. End-to-end checking is still required. Likewise, a network can filter or encrypt individual links, but only endpoint authentication and end-to-end encryption can protect the complete application exchange across all intermediate networks.

This is not a claim that the network should do nothing. Lower-layer mechanisms can be valuable performance enhancements. The principle warns against mistaking a partial in-network mechanism for an end-to-end correctness guarantee. The original argument is developed in Saltzer, Reed, and Clark's [End-to-End Arguments in System Design](https://groups.csail.mit.edu/ana/Publications/PubPDFs/End-to-End%20Arguments%20in%20System%20Design.pdf), and Internet architectural guidance appears in [RFC 1958](https://datatracker.ietf.org/doc/html/rfc1958).

#### **Fate Sharing and Soft State**

**Fate sharing** places essential communication state with the endpoint whose communication depends on it. If a router restarts, endpoint transport state should not disappear with that router; later packets may use a recovered or alternate path. This supports robustness in a network where intermediate devices can fail independently.

The Internet core is not literally stateless. Routers maintain forwarding tables, neighbor information, queues, counters, and control-plane state. The important distinction is that ordinary IP routers do not need per-connection application state to forward every datagram. [RFC 1122](https://datatracker.ietf.org/doc/rfc1122/) describes the architectural assumption that gateways forward IP datagrams independently while end-to-end reliability state resides in hosts.

**Soft state** is state that expires unless refreshed. It is useful when explicit cleanup messages may be lost. ARP/neighbor cache entries, learned routes, and NAT mappings commonly have timeouts. Expiration improves recovery from stale information, but refresh traffic and timeout selection become part of the design.

#### **Scalability, Robustness, and Evolvability**

Internet architecture repeatedly balances competing goals:

| Principle | What it enables | Trade-off or failure mode |
|---|---|---|
| Minimal common IP service | Many links and applications interoperate | Applications must add missing guarantees |
| Hierarchical aggregation | Smaller routing and forwarding state | Aggregation can constrain traffic engineering |
| End-to-end state | Intermediate failure need not destroy a connection's logic | Endpoints become more complex |
| Statistical sharing | High utilization for bursty traffic | Queueing delay and loss vary |
| Open, extensible protocols | Independent implementations and evolution | Options may be blocked by old middleboxes |
| Decentralized administration | No single operator must own the Internet | Global optimization and uniform policy are difficult |

A robust architecture is not one in which nothing fails. It is one in which failures are contained, detected, and recoverable without requiring global coordination. An evolvable architecture must also allow incremental deployment: a new protocol that requires every host, router, and operator to change simultaneously is unlikely to succeed even if its clean-slate design is elegant.


### **Standards and Protocol Evolution**

#### **IETF, RFCs, and Open Standards**

Interoperability requires specifications that independent teams can implement. The **Internet Engineering Task Force (IETF)** develops many Internet protocols through open working groups, public discussion, implementation experience, review, and rough consensus. A common path is:

1. An idea is documented as an **Internet-Draft**.
2. A working group discusses scope, design, security, operations, and interoperability.
3. Implementations and deployment experience reveal ambiguities or impractical assumptions.
4. The Internet Engineering Steering Group reviews suitable documents.
5. The RFC Editor publishes an approved document in the archival **Request for Comments (RFC)** series.

An RFC is not automatically an Internet Standard. The series also contains Best Current Practice, Informational, Experimental, and Historic documents. Status, updates, and errata matter when interpreting one. The current process is summarized by the [IETF standards-process guide](https://www.ietf.org/process/process/) and formally grounded in RFC 2026 and its updates.

Open standards reduce coordination cost: a browser, operating system, router, and server from different vendors can communicate without a private bilateral agreement. Specifications alone are insufficient, however. Interoperability testing and multiple independent implementations expose assumptions that prose may leave unclear.

#### **Interoperability and Protocol Ossification**

Successful deployment can make a protocol harder to change. A middlebox may assume that familiar fields always have familiar values, discard unknown options, or inspect information that endpoints expected to evolve. Implementations then avoid new behavior because it fails on some paths. This is **protocol ossification**: the deployed network effectively freezes details beyond the formal contract.

The response is not simply "add more extension fields." Extensions succeed only if old systems safely ignore them, endpoints can discover support, and deployment offers immediate benefit. QUIC illustrates one strategy: it runs over UDP, encrypts much of its control information, and evolves many transport functions in endpoint software. That improves deployability relative to changing TCP in every operating system and middlebox, while introducing different operational and observability trade-offs.

When evaluating a protocol change, ask four questions: Can it be deployed incrementally? Does it preserve useful behavior with old peers? Can intermediaries tolerate unknown values? Do the parties paying the deployment cost receive a benefit? Technical quality matters, but deployment mechanics often determine whether a protocol becomes real.


### **An HTTPS Request from End to End**

The following flow joins the chapter's abstractions into one concrete event. Real browsers parallelize, cache, reuse connections, race alternatives, and contact multiple services, but the stages remain useful for reasoning and troubleshooting.

![The life of an HTTPS request: local configuration, DNS resolution, hop-by-hop forwarding, secure transport, and content exchange.](assets/https-request-life.svg){fig-alt="Browser request passes through local configuration, DNS, an access router, Internet autonomous systems, and a CDN server" width="96%"}

#### **Joining the Local Network**

Before contacting a website, the device needs local reachability:

1. A link becomes available through Ethernet association, Wi-Fi authentication/association, or cellular attachment.
2. The host obtains network configuration. IPv4 commonly uses DHCP; IPv6 commonly uses Router Advertisements and may also use DHCPv6.
3. The configuration provides an address and prefix, a default gateway, and one or more DNS resolvers.
4. To send a frame to a same-link neighbor, the host maps the next-hop IP address to a link-layer address using ARP for IPv4 or Neighbor Discovery for IPv6.

The default gateway is not "the route to every destination" stored as a complete path. It is the local next hop used when no more specific local route matches. Each later router independently chooses another next hop.

A failure here often affects all remote services. Useful evidence includes whether the interface is up, whether the host has a plausible address, whether a default route exists, whether the gateway is reachable, and whether local name-server addresses were configured.

#### **Resolving a Name and Selecting a Destination**

The browser works with a name such as `www.example.com`, while IP forwarding needs a destination address. Resolution usually proceeds through several caches before a recursive DNS resolver performs additional queries. The answer may contain:

- an **A** record for IPv4;
- an **AAAA** record for IPv6;
- a CNAME chain or newer service-binding information;
- several addresses representing replicas, load balancing, or a content delivery network.

Resolution is not merely a global phone book. Answers can vary by time, client network, resolver location, service policy, and health. Modern clients may issue IPv6 and IPv4 queries asynchronously and race connection attempts so that one broken address family does not create a long user-visible pause. [RFC 8305](https://datatracker.ietf.org/doc/html/rfc8305) specifies the widely deployed Happy Eyeballs Version 2 approach; protocol work continues to evolve as DNS and transport choices change.

Authentication is still required after DNS. An address tells the client where to send packets; a valid TLS certificate and proof of private-key possession help establish that the endpoint is authorized for the requested hostname.

#### **Crossing Links, Routers, and Autonomous Systems**

The host chooses a route from its local table and encapsulates the IP datagram in a frame for the first hop. At a router:

1. The incoming link validates and delivers the frame.
2. The router removes the incoming link-layer framing.
3. It validates relevant IP fields and decreases the hop limit (IPv6) or TTL (IPv4).
4. It performs a longest-prefix match in the forwarding table.
5. It queues the datagram for an output interface.
6. It creates new link-layer framing for the next hop.

The link header therefore changes from hop to hop. The source and destination IP addresses normally identify the end-to-end datagram, although NAT can rewrite addresses and ports at an administrative boundary. Transport ports identify endpoint sockets, not routers along the path.

Within an ISP or enterprise, an interior routing protocol helps construct forwarding state. Between autonomous systems, BGP advertisements and policy influence the AS path. The forward response path may differ from the request path, so a traceroute in one direction never proves complete symmetry.

#### **Establishing Secure Transport and Exchanging Content**

For HTTP/1.1 or HTTP/2 over TCP, the client typically completes a TCP handshake and then a TLS handshake. TLS negotiates cryptographic parameters, authenticates the server certificate, derives shared traffic keys, and protects later HTTP bytes. HTTP/2 can multiplex many streams over one secured connection.

HTTP/3 uses QUIC over UDP. QUIC integrates transport and TLS 1.3 handshake behavior, supports multiple streams without TCP's cross-stream head-of-line blocking, and can evolve largely in user-space implementations. It still relies on IP and the same underlying packet path.

After secure transport exists, the browser sends an HTTP request containing a method, target, headers, and possibly a body. The selected edge or origin server returns a status, headers, and content. Rendering a page may then trigger tens or hundreds of dependent requests. Connection reuse, caching, compression, CDN placement, and request prioritization can therefore matter as much as the transfer time of the first object.

The total user-visible latency is a dependency graph rather than one number:

$$
T_{page} \approx T_{local} + T_{DNS} + T_{connect} + T_{secure} + T_{request/response} + T_{render/dependencies}.
$$

Some terms overlap because modern clients work concurrently. The expression is a diagnostic checklist, not a claim that every browser executes strictly serially.


In [ ]:
#| eval: false

import socket
import ssl
from time import perf_counter


def measure_https_head(
    host: str = "example.com",
    port: int = 443,
    timeout: float = 5.0,
) -> dict[str, object]:
    """Measure major stages of one simple HTTPS request.

    This is an educational probe, not a browser benchmark. Browsers cache DNS,
    reuse connections, negotiate HTTP/2 or HTTP/3, race addresses, and perform
    work concurrently. Run this cell only where outbound network access is
    permitted.
    """

    # Stage 1: resolve the hostname into candidate TCP endpoints.
    started = perf_counter()
    candidates = socket.getaddrinfo(
        host,
        port,
        type=socket.SOCK_STREAM,
    )
    dns_ms = (perf_counter() - started) * 1_000

    # This compact example tries the resolver's first candidate. Production
    # clients should handle failures and may race IPv6/IPv4 alternatives.
    family, socktype, proto, _, sockaddr = candidates[0]

    context = ssl.create_default_context()
    with socket.socket(family, socktype, proto) as raw_socket:
        raw_socket.settimeout(timeout)

        # Stage 2: establish the TCP transport connection.
        started = perf_counter()
        raw_socket.connect(sockaddr)
        tcp_ms = (perf_counter() - started) * 1_000

        # Stage 3: authenticate the server and derive TLS traffic keys.
        started = perf_counter()
        with context.wrap_socket(raw_socket, server_hostname=host) as tls_socket:
            tls_ms = (perf_counter() - started) * 1_000

            request = (
                f"HEAD / HTTP/1.1\r\n"
                f"Host: {host}\r\n"
                "Connection: close\r\n\r\n"
            ).encode("ascii")

            # Stage 4: send an HTTP request and wait for the first response byte.
            started = perf_counter()
            tls_socket.sendall(request)
            first_chunk = tls_socket.recv(4_096)
            time_to_first_byte_ms = (perf_counter() - started) * 1_000

            status_line = first_chunk.split(b"\r\n", 1)[0].decode(
                "iso-8859-1",
                errors="replace",
            )

            return {
                "selected_address": sockaddr[0],
                "dns_ms": round(dns_ms, 2),
                "tcp_connect_ms": round(tcp_ms, 2),
                "tls_handshake_ms": round(tls_ms, 2),
                "request_to_first_byte_ms": round(time_to_first_byte_ms, 2),
                "tls_version": tls_socket.version(),
                "status_line": status_line,
            }


# Network measurements vary on every run; compare stages, not one isolated value.
measure_https_head()


### **Observing the Packet's Journey**

The Python probe separates DNS, TCP, TLS, and first-response timing from an application's perspective. Operating-system tools expose complementary evidence. On Windows, the following sequence moves from local configuration outward:

```powershell
# Interface addresses, gateways, and configured DNS resolvers
ipconfig /all

# Name resolution and returned address records
Resolve-DnsName example.com

# Basic TCP reachability to the HTTPS service
Test-NetConnection example.com -Port 443

# Hop-limit based view of part of the forwarding path
tracert example.com

# HTTP/TLS behavior and response headers
curl.exe -I -v https://example.com/
```

No single tool gives ground truth about the entire Internet path. `tracert` depends on routers returning ICMP messages and may show timeouts even when forwarding works. DNS timing may reflect a warm cache. `curl` may negotiate a different protocol or address from a browser. Measurements are observations made from one endpoint at one time.

Use the request stages to localize failures:

| Symptom | Likely stage | Evidence to collect |
|---|---|---|
| No address or default route | Local network configuration | Interface state, DHCP/RA result, route table |
| Names fail but literal IP connectivity works | DNS | Resolver configuration, A/AAAA responses, timeout or error code |
| DNS succeeds but port 443 cannot connect | Routing, firewall, server reachability, or transport setup | Selected address, TCP result, alternate address family, path probes |
| TCP connects but certificate validation fails | TLS identity, time, trust store, or interception | Certificate chain, hostname, validity period, TLS alert |
| First byte is slow | Queueing, RTT, server work, cache miss, or upstream dependency | Stage timings, repeated samples, server traces, regional comparison |
| Transfer starts quickly but remains slow | Bottleneck rate, congestion control, loss, or receiver/application limit | Throughput, retransmissions, RTT variation, window and CPU observations |

This layered diagnosis avoids a common mistake: changing DNS when the access queue is overloaded, increasing bandwidth when propagation dominates, or blaming routing when the server certificate is invalid.


### **Comparison and Summary**

The chapter's most important distinctions are easy to collapse in casual language. Keeping them separate makes later protocols easier to learn:

| Do not confuse | Key distinction |
|---|---|
| Internet and Web | Infrastructure/protocol ecosystem versus one application family |
| Service, protocol, and API | Offered abstraction versus peer rules versus local access mechanism |
| Packet, IP datagram, and frame | General unit versus Internet-layer unit versus one-hop link-layer unit |
| Switch and router | Link-domain frame forwarding versus inter-network IP forwarding |
| Transmission and propagation delay | Time to serialize bits versus time for the signal to cross distance |
| Bandwidth, throughput, and goodput | Capacity versus delivered rate versus useful application rate |
| Hop and AS | One forwarding step versus an administrative routing domain |
| Reachability and identity | Ability to send to an address versus proof that the endpoint is authorized for a name |

A complete mental trace now looks like this:

1. An application asks for a networking service through an interface such as sockets.
2. Protocol layers encapsulate application data with the metadata needed at each scope.
3. The local link carries a frame to a next hop; routers forward the enclosed IP datagram hop by hop.
4. Packet switching statistically shares finite links, so queues and losses appear under contention.
5. End-to-end performance combines processing, queueing, serialization, propagation, transport behavior, and application dependencies.
6. Autonomous networks cooperate through open protocols and policy rather than one global controller.
7. Endpoints establish the reliability, identity, and application semantics that a best-effort IP core intentionally does not guarantee.

The remaining chapters zoom into each mechanism: links and LANs, IP forwarding, routing, transport reliability, congestion control, applications, security, operations, and programmable network infrastructure. The life of a packet is the map that keeps those details connected.
